# 4장 — 다시 뽑기: 재시도와 Best-of-N

웹 챕터(4장)와 **같은 예제, 같은 수치**를 파이썬으로 직접 계산합니다.
곱셈의 저주를 되살리는 첫 번째 길, "N번 중 한 번이라도"의 확률 $1-(1-p)^N$.

> 🤖 **AI 튜터와 함께 공부하세요**: 아래 셀을 실행하면 화면 오른쪽 아래에 튜터 버튼이 뜹니다.
> 로컬 Ollama 가 꺼져 있어도 노트북 학습에는 지장이 없습니다.

In [ ]:
from tutor import chat
chat('ch4')  # 이 챕터를 아는 AI 튜터 (로컬 Ollama — 없어도 학습엔 지장 없음)

## 1. 공식: 실패 쪽에서 세면 곱셈이 우리 편

$$P(\text{N번 중 한 번이라도 성공}) = 1 - (1-p)^N$$

워밍업의 10단계 일(성공률 35%)에 적용해 봅니다. 5번이면 88.4%, 10번이면 98.7%.

> 아래 코드의 "필요한 N" 공식에 나오는 ⌈ ⌉ 는 **올림** 기호입니다. 로그가 낯설면
> (1−p)를 거듭 곱하며 "몇 번째에 목표를 넘나"를 세어도 같은 답. 로그식은 그
> "몇 번"을 바로 구하는 지름길일 뿐이에요.

In [ ]:
import math

p = 0.35  # 워밍업의 0.9^10 ≈ 35%

def at_least_once(p, N):
    """N번 중 한 번이라도 성공할 확률 — 전부 실패(1-p)^N 을 1에서 뺀다."""
    return 1 - (1 - p) ** N

for N in [1, 2, 3, 5, 10, 20]:
    print(f'N={N:2d} → {at_least_once(p, N)*100:5.1f}%')

# 웹 랩·본문과 같은 수치인지 자가검증
assert abs(at_least_once(p, 5) - 0.8840) < 5e-4   # 5번 → 88.4%
assert abs(at_least_once(p, 10) - 0.9865) < 5e-4  # 10번 → 98.7%
assert abs(at_least_once(0.5, 2) - 0.75) < 1e-12  # 미니체크: 50% 두 번 → 75%

# 목표 성공률을 넘는 데 필요한 N = ⌈log(1-목표)/log(1-p)⌉
def needed_N(p, target):
    return math.ceil(math.log(1 - target) / math.log(1 - p))

for t in [0.5, 0.9, 0.99]:
    print(f'{t*100:.0f}% 넘기기 → N = {needed_N(p, t)}')
assert needed_N(0.35, 0.9) == 6
assert needed_N(0.35, 0.99) == 11
assert needed_N(0.5, 0.99) == 7   # 병목 개선으로 p를 50%로 올리면 11 → 7

# 기하분포 — 첫 성공까지의 평균 시도 횟수 = 1/p
expected = 1 / p
print(f'첫 성공까지 평균 {expected:.3f}회')
assert abs(expected - 2.857) < 5e-4
print('✅ 본문·랩 수치 자가검증 통과')

## 2. 수확 체감과 비용: 이득은 앞쪽에 몰려 있다

성공률 곡선(왼쪽 축)은 가파르게 오르다 평평해지고, 비용(오른쪽 축, 시도당 토큰 1,000개 가정)은
N에 정직하게 비례합니다. "무조건 크게"가 아니라 목표를 정해 N을 역산하는 이유.

In [ ]:
import matplotlib.pyplot as plt
plt.rcParams['font.family'] = 'Malgun Gothic'
plt.rcParams['axes.unicode_minus'] = False

Ns = list(range(1, 21))
succ = [at_least_once(p, N) * 100 for N in Ns]
cost = [N * 1000 for N in Ns]

fig, ax1 = plt.subplots(figsize=(8, 4.5))
ax1.plot(Ns, succ, marker='o', ms=4, color='#6366f1', label='성공률 1-(1-p)^N')
ax1.axhline(90, ls='--', lw=1, color='#94a3b8')
ax1.annotate('90% 목표 → N=6', (6, 90), textcoords='offset points', xytext=(8, -14), color='#475569', fontsize=9)
ax1.set_xlabel('시도 횟수 N')
ax1.set_ylabel('한 번이라도 성공할 확률 (%)', color='#6366f1')
ax1.set_xticks([1, 5, 6, 10, 15, 20])
ax1.grid(alpha=0.3)
ax2 = ax1.twinx()
ax2.plot(Ns, cost, marker='s', ms=3, color='#f59e0b', label='뽑기 비용 (토큰)')
ax2.set_ylabel('비용 (토큰, 시도당 1,000개 가정)', color='#f59e0b')
ax1.set_title(f'p = {p:.0%}: 성공률은 수확 체감, 비용은 정비례')
fig.legend(loc='lower right', bbox_to_anchor=(0.88, 0.15))
plt.tight_layout()
plt.show()

## 3. 기하분포: 첫 성공은 몇 번째에 나올까

$P(\text{첫 성공}=k) = (1-p)^{k-1} \cdot p$. 평균은 $1/p \approx 2.9$번째.
가장 흔한 결말은 의외로 "첫 번에 성공"(35%)입니다. 평균이 2.9라고 3번째가 최빈은 아니에요.

In [ ]:
ks = list(range(1, 13))
pmf = [(1 - p) ** (k - 1) * p for k in ks]
assert abs(pmf[0] - 0.35) < 1e-12
assert abs(pmf[1] - 0.2275) < 1e-12
assert abs(sum((1 - p) ** (k - 1) * p * k for k in range(1, 500)) - 1 / p) < 1e-6  # 기대값 = 1/p

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(ks, [v * 100 for v in pmf], color='#6366f1', alpha=0.85)
ax.axvline(1 / p, ls='--', lw=1.2, color='#f43f5e')
ax.annotate(f'평균 1/p ≈ {1/p:.1f}회', (1 / p, max(pmf) * 100), textcoords='offset points', xytext=(8, -4), color='#f43f5e', fontsize=10)
ax.set_xlabel('첫 성공이 나온 시도 번호 k')
ax.set_ylabel('확률 (%)')
ax.set_title('기하분포: p = 35%의 첫 성공 시점')
ax.set_xticks(ks)
ax.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()
print('✅ 기하분포 자가검증 통과')

## 4. 오늘의 발견

- **N번 중 한 번이라도 = 1 − (1−p)^N**: 실패의 곱을 1에서 뺀다(여사건). 35%짜리 일도 5번이면 88.4%, 10번이면 98.7%.
- **첫 성공까지 평균 1/p회**(기하분포). p=35%면 2.9회 분량의 예산 감각.
- **수확 체감 + 비용 정비례**: 목표(50/90/99%)를 정하고 N을 역산: 90%→6회, 99%→11회. 병목 개선(p 35→50%)이면 99% 목표가 11→7회로.
- **대전제**: 성공을 알아볼 수 있어야 한다. 판별의 수학은 6장(검수자), 정답을 몰라도 쓰는 길은 5장(다수결).

**웹 챕터로 돌아가 연습 10문제를 풀면 진도가 채워져요.**